# Embedding Fine-tuning with NeMo Microservices

Fine-tune an embedding model and improve retrieval by 10%+ in 75 minutes

## Prerequisites

1. **Deploy NeMo Microservices 25.8.0+:**  
   https://docs.nvidia.com/nemo/microservices/latest/get-started/setup/minikube/

2. **Register base model:**
   ```bash
   helm upgrade nemo nmp/nemo-microservices-helm-chart --reuse-values \
     --set 'customizer.customizationTargets.targets.nvidia/llama-3\.2-nv-embedqa-1b@v2.enabled=true'
   kubectl delete pod -l app.kubernetes.io/name=nemo-customizer
   ```

3. **Get HuggingFace token:** https://huggingface.co/settings/tokens

Then update the config below and run all cells.


## Overview

This notebook demonstrates the complete workflow for fine-tuning an embedding model using NeMo Microservices. You'll take a base embedding model, adapt it to scientific domain data, deploy it as a production NIM, and measure the performance improvement on a benchmark retrieval task.


## Objectives

By the end of this notebook, you will:
- Fine-tune `nvidia/llama-3.2-nv-embedqa-1b-v2` on 68K scientific paper triplets from SPECTER dataset
- Deploy the fine-tuned model as a production-ready NIM inference service
- Evaluate retrieval performance on the SciDocs benchmark
- Achieve measurable improvement: baseline recall@5 of 0.159 → ~0.176 (+10% improvement)


In [ ]:
# Install dependencies
%pip install -q datasets huggingface_hub openai nemo-microservices


In [ ]:
# Configuration - UPDATE THESE
NDS_URL = "http://data-store.test"     # Your Data Store URL
NEMO_URL = "http://nemo.test"          # Your NeMo Microservices URL
NIM_URL = "http://nim.test"            # Your NIM URL
HF_TOKEN = ""                          # Your HuggingFace token
NS = "embed-ft"                        # Namespace for this tutorial


In [ ]:
# Configuration for NVIDIA Internal Hackathon
NDS_URL = "https://datastore.aire.nvidia.com"
NEMO_URL = "https://nmp.aire.nvidia.com"
NIM_URL = "https://nim.aire.nvidia.com"
HF_TOKEN = ""
NS = "hackathon-rlempka-v2"  # Use unique namespace

In [ ]:
# Imports
from datasets import load_dataset
from nemo_microservices import NeMoMicroservices
from huggingface_hub import HfApi
from openai import OpenAI
import json, os, requests
from time import sleep


In [ ]:
# Initialize NeMo client
nemo = NeMoMicroservices(base_url=NEMO_URL, inference_base_url=NIM_URL)
print("✓ NeMo client initialized")


In [ ]:
# 1. Prepare data (5 min)
print("Downloading SPECTER dataset...")
os.environ["HF_TOKEN"] = HF_TOKEN
data = load_dataset("embedding-data/SPECTER")['train'].shuffle(seed=42).select(range(68400))

print("Splitting dataset...")
splits = data.train_test_split(test_size=0.10, seed=42)
val = splits['test'].train_test_split(test_size=0.50, seed=42)['train']

print("Saving to JSONL...")
os.makedirs("data", exist_ok=True)
for name, d in [("train", splits['train']), ("val", val)]:
    with open(f"data/{name}.jsonl", "w") as f:
        for row in d:
            f.write(json.dumps({"query": row['set'][0], "pos_doc": row['set'][1], "neg_doc": [row['set'][2]]}) + "\n")

print(f"✓ Data prepared: {len(splits['train'])} train, {len(val)} val")


## Step 1: Prepare Data

Download the SPECTER dataset and format it for embedding fine-tuning.


In [ ]:
# 2. Upload data to NeMo (3 min)
print("Creating namespace...")
nemo.namespaces.create(id=NS)
requests.post(f"{NDS_URL}/v1/datastore/namespaces", data={"namespace": NS})

print("Creating data repository...")
hf = HfApi(endpoint=f"{NDS_URL}/v1/hf", token=None)
hf.create_repo(f"{NS}/data", repo_type='dataset')

print("Uploading files...")
hf.upload_file(path_or_fileobj="data/train.jsonl", path_in_repo="training/training.jsonl", repo_id=f"{NS}/data", repo_type='dataset')
hf.upload_file(path_or_fileobj="data/val.jsonl", path_in_repo="validation/validation.jsonl", repo_id=f"{NS}/data", repo_type='dataset')

print("Registering dataset...")
nemo.datasets.create(name="data", namespace=NS, files_url=f"hf://datasets/{NS}/data")

print("✓ Data uploaded")


## Step 2: Upload to NeMo

Upload the prepared data to NeMo Data Store and register it for training.


In [ ]:
# 3. Train model (45 min)
print("Creating training config...")
nemo.customization.configs.create(
    name="cfg@v1", 
    namespace=NS, 
    target="nvidia/llama-3.2-nv-embedqa-1b@v2", 
    training_options=[{"training_type": "sft", "finetuning_type": "all_weights", "num_gpus": 1, "micro_batch_size": 8}], 
    max_seq_length=2048)

In [59]:
print("Starting training job...")
job = nemo.customization.jobs.create(
    name="job", 
    config=f"{NS}/cfg@v1", 
    dataset={"namespace": NS, "name": "data"},
    hyperparameters={"finetuning_type": "all_weights", "epochs": 1, "batch_size": 256, "learning_rate": 5e-6}, 
    output_model=f"{NS}/model")

print(f"Training job: {job.id}")
print("⏱️  This takes ~45 minutes...")

while nemo.customization.jobs.retrieve(job.id).status in ["pending", "created", "running"]: 
    sleep(30)

status = nemo.customization.jobs.retrieve(job.id).status
print(f"✓ Training complete: {status}")


Starting training job...
Training job: cust-3UZC1J3nw5MV8guMUxvt4g
⏱️  This takes ~45 minutes...
✓ Training complete: completed


## Step 3: Train Model

Fine-tune the embedding model using supervised contrastive learning (~45 minutes).


In [ ]:
# 4. Deploy model (10 min)
print("Deploying model...")

# Create deployment with INLINE config (don't reference named config)
nemo.deployment.model_deployments.create(
    name="nim",
    namespace=NS,
    config={
        "model": f"{NS}/model@{job.id}",
        "nim_deployment": {
            "image_name": "nvcr.io/nim/nvidia/llama-3.2-nv-embedqa-1b-v2",
            "image_tag": "1.6.0",
            "gpu": 1,
            "disable_lora_support": True
        }
    }
)

print("⏱️  Waiting for deployment (~10 min)...")
while nemo.deployment.model_deployments.retrieve("nim", NS).status_details.status != 'ready':
    sleep(10)

print("✓ Model deployed")

Creating deployment config...
Deploying model...


NotFoundError: Error code: 404 - {'message': 'deployment config not found', 'requestId': 'b43d1274-d65a-9a29-bdfa-50cbe2a31d9f'}

## Step 4: Deploy Model

Deploy the fine-tuned model as a NIM inference service (~10 minutes).


In [73]:
# 5. Test inference
client = OpenAI(base_url=f"{NIM_URL}/v1", api_key="None")
emb = client.embeddings.create(
    input=["Deep learning for computer vision"], 
    model=f"{NS}/model", 
    extra_body={"input_type": "query"})

print(f"✓ Inference works! Embedding dimension: {len(emb.data[0].embedding)}")


✓ Inference works! Embedding dimension: 2048


## Step 5: Test Inference

Verify the deployed model responds to embedding requests.


In [74]:
# 6. Evaluate on SciDocs (12 min)
print("Creating evaluation config...")

eval_cfg = {
    "type": "retriever",
    "namespace": NS,
    "tasks": {
        "scidocs": {
            "type": "beir",
            "dataset": {"files_url": "file://scidocs/"},
            "metrics": {"recall_5": {"type": "recall_5"}}}}}

target = {
    "type": "retriever",
    "retriever": {
        "pipeline": {
            "query_embedding_model": {
                "api_endpoint": {"url": f"{NIM_URL}/v1/embeddings", "model_id": f"{NS}/model"}},
            "index_embedding_model": {
                "api_endpoint": {"url": f"{NIM_URL}/v1/embeddings", "model_id": f"{NS}/model"}},
            "top_k": 10}}}

print("Running evaluation...")
ejob = nemo.evaluation.jobs.create(config=eval_cfg, target=target)

print("⏱️  This takes ~12 minutes...")
while nemo.evaluation.jobs.retrieve(ejob.id).status in ["pending", "created", "running"]:
    sleep(30)

res = nemo.evaluation.jobs.results(ejob.id)
print("✓ Evaluation complete")

Creating evaluation config...
Running evaluation...
⏱️  This takes ~12 minutes...


NotFoundError: Error code: 404 - {'detail': 'Job results not available.'}

In [79]:
# 6. Evaluate on SciDocs (12 min)
print("Creating evaluation config...")

# Use internal NIM proxy (workaround for HTTPS → HTTP mapping issue)
EVAL_NIM_URL = "http://nemo-nim-proxy:8000"
print(f"Using URL: {EVAL_NIM_URL}")

eval_cfg = {
    "type": "retriever",
    "namespace": NS,
    "tasks": {
        "scidocs": {
            "type": "beir",
            "dataset": {"files_url": "file://scidocs/"},
            "metrics": {"recall_5": {"type": "recall_5"}}}}}

target = {
    "type": "retriever",
    "retriever": {
        "pipeline": {
            "query_embedding_model": {
                "api_endpoint": {"url": f"{EVAL_NIM_URL}/v1/embeddings", "model_id": f"{NS}/model"}},
            "index_embedding_model": {
                "api_endpoint": {"url": f"{EVAL_NIM_URL}/v1/embeddings", "model_id": f"{NS}/model"}},
            "top_k": 10}}}

print("Running evaluation...")
ejob = nemo.evaluation.jobs.create(config=eval_cfg, target=target)

print("⏱️  This takes ~12 minutes...")
while nemo.evaluation.jobs.retrieve(ejob.id).status in ["pending", "created", "running"]:
    sleep(30)

final_status = nemo.evaluation.jobs.retrieve(ejob.id).status
if final_status == 'completed':
    res = nemo.evaluation.jobs.results(ejob.id)
    print("✓ Evaluation complete")
else:
    print(f"❌ Status: {final_status}")

Creating evaluation config...
Using URL: http://nemo-nim-proxy:8000
Running evaluation...
⏱️  This takes ~12 minutes...
✓ Evaluation complete


In [81]:
print("Available tasks:", res.tasks.keys())
print("\nSciDocs metrics:", res.tasks['scidocs'].metrics.keys())
print("\nFull metrics:", res.tasks['scidocs'].metrics)

Available tasks: dict_keys(['scidocs'])

SciDocs metrics: dict_keys(['retriever.recall_5'])

Full metrics: {'retriever.recall_5': MetricResult(scores={'recall_5': Score(value=0.1699499999999998, stats=ScoreStats(count=None, max=None, mean=None, min=None, stddev=None, stderr=None, sum=None, sum_squared=None, variance=None))})}


## Step 6: Evaluate Performance

Run the SciDocs benchmark to measure retrieval quality (~12 minutes).


In [82]:
# 7. Display results
baseline = 0.159
yours = res.tasks['scidocs'].metrics['retriever.recall_5'].scores['recall_5'].value
improvement = ((yours / baseline) - 1) * 100

print("="*70)
print("EVALUATION RESULTS - SciDocs Benchmark")
print("="*70)
print(f"Baseline (pretrained): {baseline:.3f}")
print(f"Fine-tuned model:      {yours:.3f}")
print(f"Improvement:          +{improvement:.1f}%")
print("="*70)
print(f"\nModel deployed at: {NIM_URL}/v1/embeddings")
print(f"Model name: {NS}/model")

EVALUATION RESULTS - SciDocs Benchmark
Baseline (pretrained): 0.159
Fine-tuned model:      0.170
Improvement:          +6.9%

Model deployed at: https://nim.aire.nvidia.com/v1/embeddings
Model name: hackathon-rlempka-v2/model


## Summary

**What You Built:**
- Fine-tuned embedding model optimized for scientific paper retrieval
- Production NIM deployment serving embeddings via OpenAI-compatible API
- Measured 10%+ improvement in recall@5 on zero-shot SciDocs benchmark

**Key Results:**
- Baseline: 0.159 recall@5
- Fine-tuned: ~0.176 recall@5
- **Impact:** Your model finds relevant papers in top-5 results 10% more often


## Next Steps

**Scale Up:**
- Train on full SPECTER dataset (684K examples) for 15-20% improvement
- Increase to 3 epochs for better convergence
- Use multi-GPU training for faster iterations

**Apply to Your Domain:**
- Format your data as query-positive-negative triplets
- Replace SPECTER dataset with your domain data (legal, medical, product catalogs, etc.)
- Evaluate on your own retrieval tasks

**Production Deployment:**
- Set up monitoring with Prometheus/Grafana
- Implement A/B testing against baseline model
- Scale NIMs horizontally for high-traffic scenarios
- Add request logging and performance metrics

**Learn More:**
- [NeMo Microservices Documentation](https://docs.nvidia.com/nemo/microservices/latest/)
- [Embedding Model Guide](https://build.nvidia.com/nvidia/llama-3_2-nv-embedqa-1b-v2)
- [Other NeMo Tutorials](../../../README.md)


## Cleanup (Optional)

Run the cell below to delete all resources created in this tutorial.


In [ ]:
# CLEANUP (optional - run to delete all resources)
print("Deleting deployment...")
nemo.deployment.model_deployments.delete(deployment_name="nim", namespace=NS)

print("Deleting models...")
for m in nemo.models.list(filter={"namespace": NS}).data:
    nemo.models.delete(namespace=NS, model_name=m.name.split('/')[-1])

print("Deleting dataset...")
nemo.datasets.delete(namespace=NS, dataset_name="data")

print("Deleting data repo...")
hf.delete_repo(f"{NS}/data", repo_type='dataset')

print("Deleting configs...")
nemo.deployment.configs.delete(config_name="dcfg", namespace=NS)
nemo.customization.configs.delete(config_name="cfg", namespace=NS)

print("✓ Cleanup complete")

In [56]:

response = requests.get(f"{NEMO_URL}/v1/customization/targets")
if response.ok:
    targets = response.json()
    for target in targets.get('data', []):
        name = target.get('name', 'unknown')
        enabled = target.get('enabled', False)
        status = target.get('status', 'unknown')
        print(f"- {name} (enabled: {enabled}, status: {status})")


- nemotron-nano-llama-3.1-8b@1.0 (enabled: True, status: ready)
- llama-3.2-1b-instruct@2.0 (enabled: True, status: ready)
- llama-3.1-8b-instruct@2.0 (enabled: True, status: ready)


In [57]:
!echo $NEMO_URL

https://nmp.aire.nvidia.com
